# 4. Polarity analysis (Section 4.3, Table 3)

Uses the LeIA sentiment score already present in the datasets (`score_sentiment`, computed by the
collection platform). Negative < -0.05, neutral [-0.05, 0.05], positive > 0.05.

In [ ]:
from utils import *

In [ ]:
wa = load_dataset("whatsapp")["score_sentiment"].dropna().to_numpy()
tg = load_dataset("telegram")["score_sentiment"].dropna().to_numpy()

def categories(s):
    return np.array([(s < -0.05).sum(), ((s >= -0.05) & (s <= 0.05)).sum(), (s > 0.05).sum()])

def describe(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    neg, neu, pos = 100 * categories(s) / len(s)
    return {"messages": len(s), "mean": s.mean(), "sd": s.std(ddof=1), "median": median,
            "Q1": q1, "Q3": q3, "IQR": q3 - q1,
            "negative %": neg, "neutral %": neu, "positive %": pos}

table3 = pd.DataFrame({"FW.Br": describe(wa), "FT.Br": describe(tg)}).round(3)
table3.to_csv(results_folder("04_polarity") / "table3_polarity.csv")
table3

## Tests

In [ ]:
U, p_mw = stats.mannwhitneyu(wa, tg, alternative="two-sided", method="asymptotic")
delta, (ci_low, ci_high) = cliffs_delta(wa, tg)   # positive = FW.Br more positive

counts = np.vstack([categories(wa), categories(tg)])
chi2, p_chi2, dof, _ = stats.chi2_contingency(counts)
cramers_v = np.sqrt(chi2 / (counts.sum() * (min(counts.shape) - 1)))
neg_diff = 100 * (counts[1, 0] / counts[1].sum() - counts[0, 0] / counts[0].sum())

print(f"Mann-Whitney U = {U:,.0f}, p = {p_mw:.1e}")
print(f"Cliff's delta = {delta:.3f}, 95% CI [{ci_low:.3f}, {ci_high:.3f}]")
print(f"chi2({dof}) = {chi2:.2f}, p = {p_chi2:.1e}, Cramer's V = {cramers_v:.3f}")
print(f"negative messages: +{neg_diff:.1f} percentage points in FT.Br")

## Robustness: group as the unit of analysis (footnote in Section 4.3)
Median score per group; only groups with at least 10 (or 30) messages.

In [ ]:
group_p = {}
for minimum in [10, 30]:
    medians = []
    for platform in ["whatsapp", "telegram"]:
        g = load_dataset(platform).dropna(subset=["score_sentiment"]) \
              .groupby("id_group_anonymous")["score_sentiment"].agg(["median", "size"])
        medians.append(g.loc[g["size"] >= minimum, "median"])
    group_p[minimum] = stats.mannwhitneyu(*medians, alternative="two-sided").pvalue
    print(f"groups with >= {minimum} messages: {len(medians[0])} FW.Br, {len(medians[1])} FT.Br, p = {group_p[minimum]:.2f}")

## Comparison with the paper

In [ ]:
compare_with_paper([
    ("FW.Br mean", 0.270, table3.loc["mean", "FW.Br"]), ("FT.Br mean", 0.230, table3.loc["mean", "FT.Br"]),
    ("FW.Br sd", 0.551, table3.loc["sd", "FW.Br"]), ("FT.Br sd", 0.570, table3.loc["sd", "FT.Br"]),
    ("FW.Br median", 0.365, table3.loc["median", "FW.Br"]), ("FT.Br median", 0.302, table3.loc["median", "FT.Br"]),
    ("FW.Br negative %", 25.4, round(table3.loc["negative %", "FW.Br"], 1)),
    ("FT.Br negative %", 28.4, round(table3.loc["negative %", "FT.Br"], 1)),
    ("FW.Br positive %", 64.8, round(table3.loc["positive %", "FW.Br"], 1)),
    ("FT.Br positive %", 60.8, round(table3.loc["positive %", "FT.Br"], 1)),
    ("Mann-Whitney U", 22820276, U), ("Cliff's delta", 0.036, round(delta, 3)),
    ("Cliff's delta CI low", 0.016, round(ci_low, 3)), ("Cliff's delta CI high", 0.056, round(ci_high, 3)),
    ("chi2", 21.28, round(chi2, 2)), ("Cramer's V", 0.039, round(cramers_v, 3)),
    ("group-level p (>= 10 messages)", 0.61, round(group_p[10], 2)),
], tolerance=0.5)